# Classroom Detection - 4 Class RT-DETR (Kaggle GPU)

**Classes:** Board, Desk, Chair, Fan (4 classes)  
**Total Images:** ~2,377 images  
**Training Time:** ~4 hours on T4 x2  
**Non-COCO Classes:** Board, Fan (2 non-COCO!)  

---

### Before running:
1. In right panel > **Session options** > set **Accelerator = GPU T4 x2**
2. Make sure your dataset (`classroom-detection`) is attached under **Input**
3. Click **Run All**

## 1 - Setup GPU & Packages

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# Install packages
!pip install -q ultralytics opencv-python pillow pyyaml tqdm

from ultralytics import RTDETR
print("ultralytics imported successfully")


## 2 - Inspect Kaggle Input Dataset

In [ ]:
import os
from pathlib import Path

# Kaggle datasets are at /kaggle/input/<dataset-slug>/
print("=== /kaggle/input contents ===")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_dir():
        imgs = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        if imgs:
            print(f"  {p.relative_to('/kaggle/input')} -> {len(imgs)} images")


## 3 - Merge All 4 Classes into YOLO Dataset

In [ ]:
import shutil
import random
import yaml
from pathlib import Path
from collections import defaultdict

random.seed(42)

# Output to /kaggle/working
proc_dir = Path("/kaggle/working/data/processed")
if proc_dir.exists():
    shutil.rmtree(proc_dir)

for split in ["train", "val", "test"]:
    (proc_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (proc_dir / split / "labels").mkdir(parents=True, exist_ok=True)

# Gather all image-label pairs from /kaggle/input
input_root = Path("/kaggle/input")
all_pairs = []

for img in input_root.rglob("*.jpg"):
    # Find matching label in sibling 'labels' folder
    lbl_cand1 = img.parent.parent / "labels" / f"{img.stem}.txt"
    lbl_cand2 = img.parent / f"{img.stem}.txt"
    lbl = None
    if lbl_cand1.exists():
        lbl = lbl_cand1
    elif lbl_cand2.exists():
        lbl = lbl_cand2
    
    if lbl:
        # Determine class category from path
        path_str = str(img).lower()
        cat = "object"
        for cls in ["board", "chair", "desk", "fan"]:
            if cls in path_str:
                cat = cls
                break
        all_pairs.append((img, lbl, cat))

# Count by category
by_cat = defaultdict(list)
for img, lbl, cat in all_pairs:
    by_cat[cat].append((img, lbl))

print(f"Total paired samples found: {len(all_pairs)}")
for cat, pairs in sorted(by_cat.items()):
    print(f"  {cat}: {len(pairs)} samples")

# Shuffle and split 70 / 20 / 10
random.shuffle(all_pairs)
n = len(all_pairs)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)

splits = {
    "train": all_pairs[:n_train],
    "val":   all_pairs[n_train:n_train + n_val],
    "test":  all_pairs[n_train + n_val:]
}

# Copy into /kaggle/working/data/processed
for split_name, samples in splits.items():
    for idx, (img_p, lbl_p, cat) in enumerate(samples):
        new_stem = f"{cat}_{idx:05d}"
        shutil.copy2(img_p, proc_dir / split_name / "images" / f"{new_stem}{img_p.suffix}")
        shutil.copy2(lbl_p, proc_dir / split_name / "labels" / f"{new_stem}.txt")
    print(f"  {split_name}: {len(samples)} samples copied")

# Auto-detect classes from found categories
found_classes = sorted([c for c in by_cat.keys() if c != "object"])
if not found_classes:
    found_classes = ["board", "chair", "desk", "fan"]

# Write data.yaml
data_cfg = {
    "path": "/kaggle/working/data/processed",
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(found_classes),
    "names": found_classes
}

yaml_path = proc_dir / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\ndata.yaml written: {yaml_path}")
print(f"Classes ({len(found_classes)}): {found_classes}")
print(f"Non-COCO: {[c for c in found_classes if c in ['board','fan']]}")


## 4 - Train RT-DETR-L (~4 hours on T4 x2, 30 epochs)

In [ ]:
from ultralytics import RTDETR

print("=" * 70)
print("RT-DETR-L TRAINING  |  T4 x2  |  30 epochs  |  ~4 hours")
print("=" * 70)

model = RTDETR("rtdetr-l.pt")

results = model.train(
    data="/kaggle/working/data/processed/data.yaml",
    epochs=30,
    imgsz=640,
    batch=32,         # 16 per GPU x 2 GPUs
    device=[0, 1],    # T4 x2
    patience=15,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.01,
    warmup_epochs=3,
    project="/kaggle/working/runs/train",
    name="classroom_4class",
    save_period=5,
    seed=42,
    deterministic=True,
    pretrained=True,
    workers=4,
    verbose=True,
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)


## 5 - Evaluate on Test Set

In [ ]:
from ultralytics import RTDETR

best_pt = "/kaggle/working/runs/train/classroom_4class/weights/best.pt"

eval_model = RTDETR(best_pt)

metrics = eval_model.val(
    data="/kaggle/working/data/processed/data.yaml",
    split="test",
    conf=0.5,
    iou=0.5,
    save_json=True,
    plots=True
)

print("\n" + "=" * 70)
print("4-CLASS MODEL EVALUATION RESULTS")
print("=" * 70)
print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")
print("=" * 70)
print("\nPer-class results:")
for i, name in eval_model.names.items():
    r = metrics.box.class_result(i)
    print(f"  {name:10s}  P={r[0]:.3f}  R={r[1]:.3f}  mAP@0.5={r[2]:.3f}")


## 6 - Download Weights

In [ ]:
import shutil
from pathlib import Path

# Package best.pt + plots into a single zip for download
out_dir = Path("/kaggle/working/output_4class")
out_dir.mkdir(exist_ok=True)

shutil.copy2(
    "/kaggle/working/runs/train/classroom_4class/weights/best.pt",
    out_dir / "best.pt"
)

# Copy training plots
for png in Path("/kaggle/working/runs/train/classroom_4class").glob("*.png"):
    shutil.copy2(png, out_dir / png.name)

shutil.make_archive("/kaggle/working/classroom_4class_model", "zip", str(out_dir))

print("Output zip ready: /kaggle/working/classroom_4class_model.zip")
print("  -> In Kaggle: click the file in the output panel and download")
